# Classification des Cultures Agricoles

Ce notebook utilise les indices NDVI et EVI pour identifier les types de cultures.

In [ ]:
!pip install geemap earthengine-api rasterio matplotlib -q
import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.2, -4.8, 15.4, -4.6])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B4', 'B8']), 'input.tif', scale=30, region=roi)

In [ ]:
with rasterio.open('input.tif') as src: data = src.read().astype(np.float32)
blue, red, nir = data[0], data[1], data[2]
ndvi = (nir - red) / (nir + red + 1e-8)
evi = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))

crop_idx = (ndvi > 0.5) * evi
plt.imshow(crop_idx, cmap='YlGn')
plt.title("Classification des Zones de Cultures")
plt.show()